# Committee Sampling Simulation Analysis

This notebook analyzes simulation logs from committee sampling protocol runs.
It extracts graph properties, messaging statistics, and committee consensus metrics.

## 1. Setup and Imports

In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Set, Tuple, Any

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline


## 2. Configuration

In [ ]:
LOG_DIRECTORY = "./logs"  # Change this to your log directory path
print(f"Analyzing logs from: {LOG_DIRECTORY}")


## 3. Utility Functions

In [ ]:
def parse_log_file(filepath: Path) -> List[Dict[str, Any]]:
    """Parse a log file containing JSON lines."""
    entries = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                if line.startswith('{'):
                    entry = json.loads(line)
                    entries.append(entry)
            except json.JSONDecodeError:
                continue
    return entries

def extract_node_id_from_logs(entries: List[Dict]) -> str:
    """Extract node ID from log entries."""
    for entry in entries:
        if 'node_id' in entry:
            return entry['node_id']
    return None

def load_all_logs(log_dir: str) -> Dict[str, List[Dict]]:
    """Load all node log files from directory."""
    log_path = Path(log_dir)
    log_files = sorted(log_path.glob('node-*.log'))
    
    all_logs = {}
    for log_file in log_files:
        entries = parse_log_file(log_file)
        node_id = extract_node_id_from_logs(entries)
        if node_id:
            all_logs[node_id] = entries
    
    print(f"Loaded logs from {len(all_logs)} nodes")
    return all_logs


## 4. Load Log Data

In [ ]:
all_logs = load_all_logs(LOG_DIRECTORY)
print(f"Total nodes: {len(all_logs)}")


## 5. Graph Analysis

In [ ]:
def extract_graph_structure(all_logs: Dict[str, List[Dict]]) -> nx.DiGraph:
    """Extract the directed graph from final neighbor lists."""
    G = nx.DiGraph()
    
    for node_id, entries in all_logs.items():
        G.add_node(node_id)
        
        for entry in entries:
            if entry.get('msg') == 'Final neighbor list':
                neighbors = entry.get('neighbors', [])
                for neighbor in neighbors:
                    G.add_edge(node_id, neighbor)
                break
    
    return G

def compute_graph_metrics(G: nx.DiGraph) -> Dict[str, Any]:
    """Compute comprehensive graph metrics."""
    metrics = {}
    
    metrics['num_nodes'] = G.number_of_nodes()
    metrics['num_edges'] = G.number_of_edges()
    
    out_degrees = [d for n, d in G.out_degree()]
    in_degrees = [d for n, d in G.in_degree()]
    
    metrics['avg_out_degree'] = np.mean(out_degrees)
    metrics['avg_in_degree'] = np.mean(in_degrees)
    metrics['std_out_degree'] = np.std(out_degrees)
    metrics['std_in_degree'] = np.std(in_degrees)
    
    one_directional = 0
    bidirectional = 0
    for u, v in G.edges():
        if G.has_edge(v, u):
            bidirectional += 1
        else:
            one_directional += 1
    
    metrics['one_directional_edges'] = one_directional
    metrics['bidirectional_edges'] = bidirectional // 2
    
    metrics['is_strongly_connected'] = nx.is_strongly_connected(G)
    
    scc = list(nx.strongly_connected_components(G))
    metrics['num_strongly_connected_components'] = len(scc)
    metrics['largest_scc_size'] = len(max(scc, key=len)) if scc else 0
    
    if metrics['is_strongly_connected']:
        try:
            metrics['diameter'] = nx.diameter(G)
        except:
            metrics['diameter'] = None
    else:
        largest_scc = max(scc, key=len) if scc else set()
        if len(largest_scc) > 1:
            subgraph = G.subgraph(largest_scc)
            try:
                metrics['diameter_largest_scc'] = nx.diameter(subgraph)
            except:
                metrics['diameter_largest_scc'] = None
        else:
            metrics['diameter_largest_scc'] = None
    
    return metrics

print("Building graph from neighbor lists...")
graph = extract_graph_structure(all_logs)
graph_metrics = compute_graph_metrics(graph)

print("\n=== GRAPH STATISTICS ===")
print(f"Number of nodes: {graph_metrics['num_nodes']}")
print(f"Number of edges: {graph_metrics['num_edges']}")
print(f"Average out-degree: {graph_metrics['avg_out_degree']:.2f} ± {graph_metrics['std_out_degree']:.2f}")
print(f"Average in-degree: {graph_metrics['avg_in_degree']:.2f} ± {graph_metrics['std_in_degree']:.2f}")
print(f"One-directional edges: {graph_metrics['one_directional_edges']}")
print(f"Bidirectional edges: {graph_metrics['bidirectional_edges']}")
print(f"Is strongly connected: {graph_metrics['is_strongly_connected']}")
print(f"Number of strongly connected components: {graph_metrics['num_strongly_connected_components']}")
print(f"Largest SCC size: {graph_metrics['largest_scc_size']}")
if 'diameter' in graph_metrics and graph_metrics['diameter']:
    print(f"Diameter: {graph_metrics['diameter']}")
elif 'diameter_largest_scc' in graph_metrics and graph_metrics['diameter_largest_scc']:
    print(f"Diameter (largest SCC): {graph_metrics['diameter_largest_scc']}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

out_degrees = [d for n, d in graph.out_degree()]
in_degrees = [d for n, d in graph.in_degree()]

axes[0].hist(out_degrees, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Out-Degree')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Out-Degree Distribution')
axes[0].axvline(np.mean(out_degrees), color='r', linestyle='--', label=f'Mean: {np.mean(out_degrees):.2f}')
axes[0].legend()

axes[1].hist(in_degrees, bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('In-Degree')
axes[1].set_ylabel('Frequency')
axes[1].set_title('In-Degree Distribution')
axes[1].axvline(np.mean(in_degrees), color='r', linestyle='--', label=f'Mean: {np.mean(in_degrees):.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Protocol Messaging Statistics

In [ ]:
def extract_mdag_statistics(all_logs: Dict[str, List[Dict]]) -> Dict[str, pd.DataFrame]:
    """Extract MDAG messaging statistics per protocol."""
    mdag_stats = {'ExAnte': [], 'ExPost': []}
    
    for node_id, entries in all_logs.items():
        for entry in entries:
            if entry.get('msg') == 'Message counts':
                logger = entry.get('logger', '')
                protocol_type = None
                
                if 'exanteMDAG' in logger:
                    protocol_type = 'ExAnte'
                elif 'expostMDAG' in logger:
                    protocol_type = 'ExPost'
                
                if protocol_type:
                    valid_msgs = entry.get('valid_messages', [])
                    total_msgs = entry.get('total_messages', [])
                    
                    for round_idx, (valid, total) in enumerate(zip(valid_msgs, total_msgs)):
                        mdag_stats[protocol_type].append({
                            'node_id': node_id,
                            'round': round_idx,
                            'valid_messages': valid,
                            'total_messages': total,
                            'acceptance_rate': valid / total if total > 0 else 0
                        })
    
    return {k: pd.DataFrame(v) for k, v in mdag_stats.items() if v}

def extract_protocol_messages(all_logs: Dict[str, List[Dict]]) -> Dict[str, pd.DataFrame]:
    """Extract ExAnte and ExPost protocol message counts."""
    protocol_msgs = {'ExAnte': [], 'ExPost': []}
    
    for node_id, entries in all_logs.items():
        for entry in entries:
            msg = entry.get('msg', '')
            
            if 'ExPost: Received message' in msg:
                protocol_msgs['ExPost'].append({
                    'node_id': node_id,
                    'round': entry.get('round', 0),
                    'from': entry.get('from', ''),
                    'sender_id': entry.get('sender_id', '')
                })
            elif 'ExAnte: Received message' in msg:
                protocol_msgs['ExAnte'].append({
                    'node_id': node_id,
                    'round': entry.get('round', 0),
                    'from': entry.get('from', ''),
                    'sender_id': entry.get('sender_id', '')
                })
    
    return {k: pd.DataFrame(v) for k, v in protocol_msgs.items() if v}

print("Extracting MDAG statistics...")
mdag_stats = extract_mdag_statistics(all_logs)

print("Extracting protocol message counts...")
protocol_msgs = extract_protocol_messages(all_logs)


In [ ]:
print("\n=== MDAG MESSAGING STATISTICS ===")

for protocol_name, df in mdag_stats.items():
    if df.empty:
        print(f"\n{protocol_name} MDAG: No data available")
        continue
    
    print(f"\n{protocol_name} MDAG:")
    
    per_round = df.groupby('round').agg({
        'valid_messages': ['mean', 'std', 'min', 'max', 'sum'],
        'total_messages': ['mean', 'std', 'min', 'max', 'sum'],
        'acceptance_rate': 'mean'
    }).round(2)
    
    print(per_round)
    print(f"\nOverall acceptance rate: {df['acceptance_rate'].mean():.2%}")


In [ ]:
if mdag_stats:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    for idx, (protocol_name, df) in enumerate(mdag_stats.items()):
        if df.empty:
            continue
        
        per_round = df.groupby('round')['valid_messages'].agg(['mean', 'std', 'min', 'max'])
        
        ax = axes[idx, 0]
        ax.plot(per_round.index, per_round['mean'], 'o-', label='Mean')
        ax.fill_between(per_round.index, 
                         per_round['mean'] - per_round['std'], 
                         per_round['mean'] + per_round['std'], 
                         alpha=0.3)
        ax.plot(per_round.index, per_round['min'], '--', label='Min', alpha=0.5)
        ax.plot(per_round.index, per_round['max'], '--', label='Max', alpha=0.5)
        ax.set_xlabel('Round')
        ax.set_ylabel('Valid Messages')
        ax.set_title(f'{protocol_name} MDAG: Valid Messages per Round')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        acceptance_rate = df.groupby('round')['acceptance_rate'].mean()
        ax = axes[idx, 1]
        ax.plot(acceptance_rate.index, acceptance_rate.values, 'o-', color='green')
        ax.set_xlabel('Round')
        ax.set_ylabel('Acceptance Rate')
        ax.set_title(f'{protocol_name} MDAG: Message Acceptance Rate')
        ax.set_ylim([0, 1.05])
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No MDAG statistics available for visualization.")


In [ ]:
print("\n=== PROTOCOL MESSAGE COUNTS ===")

for protocol_name, df in protocol_msgs.items():
    if df.empty:
        print(f"\n{protocol_name}: No data available")
        continue
    
    print(f"\n{protocol_name}:")
    per_round = df.groupby('round').size().reset_index(name='message_count')
    print(per_round.describe())


## 7. Committee Analysis

In [ ]:
def parse_committee_output(log_file: Path) -> List[Dict[str, Any]]:
    """Parse committee election output from plain text logs."""
    committee = []
    with open(log_file, 'r') as f:
        lines = f.readlines()
    
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('ID:'):
            member_id = line.split('ID:')[1].strip()
            vk = ''
            grade = 0
            
            if i + 1 < len(lines) and lines[i + 1].strip().startswith('VK:'):
                vk = lines[i + 1].strip().split('VK:')[1].strip()
            
            if i + 2 < len(lines) and lines[i + 2].strip().startswith('Grade:'):
                grade_str = lines[i + 2].strip().split('Grade:')[1].strip()
                try:
                    grade = int(grade_str)
                except ValueError:
                    grade = 0
            
            committee.append({
                'id': member_id,
                'vk': vk,
                'grade': grade
            })
            i += 3
        else:
            i += 1
    
    return committee

def extract_committees_from_json(all_logs: Dict[str, List[Dict]]) -> Dict[str, List[Dict]]:
    """Extract committee election results from JSON log entries."""
    committees = {}
    
    for node_id, entries in all_logs.items():
        for entry in entries:
            if entry.get('msg') == 'Committee elected':
                committee_data = entry.get('committee', [])
                if committee_data:
                    committees[node_id] = committee_data
                break
    
    return committees

def extract_committees_from_files(log_dir: str) -> Dict[str, List[Dict]]:
    """Extract committees from log files (trying both JSON and text parsing)."""
    committees = extract_committees_from_json(all_logs)
    
    if not committees:
        log_path = Path(log_dir)
        log_files = sorted(log_path.glob('node-*.log'))
        
        for log_file in log_files:
            committee = parse_committee_output(log_file)
            if committee:
                entries = parse_log_file(log_file)
                node_id = extract_node_id_from_logs(entries)
                if node_id:
                    committees[node_id] = committee
    
    return committees

print("Extracting committee election results...")
committees = extract_committees_from_files(LOG_DIRECTORY)
print(f"Found committee data for {len(committees)} nodes")


In [ ]:
if committees:
    committee_sizes = [len(c) for c in committees.values()]
    
    print("\n=== COMMITTEE STATISTICS ===")
    print(f"Number of nodes with committee data: {len(committees)}")
    print(f"Committee size - Min: {min(committee_sizes)}")
    print(f"Committee size - Max: {max(committee_sizes)}")
    print(f"Committee size - Mean: {np.mean(committee_sizes):.2f}")
    print(f"Committee size - Median: {np.median(committee_sizes):.2f}")
    print(f"Committee size - Std Dev: {np.std(committee_sizes):.2f}")
    
    plt.figure(figsize=(10, 6))
    plt.hist(committee_sizes, bins=20, edgecolor='black', alpha=0.7)
    plt.xlabel('Committee Size')
    plt.ylabel('Number of Nodes')
    plt.title('Distribution of Committee Sizes Across Nodes')
    plt.axvline(np.mean(committee_sizes), color='r', linestyle='--', 
                label=f'Mean: {np.mean(committee_sizes):.2f}')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("\nNo committee data found in logs.")


## 8. Committee Consensus Analysis

In [ ]:
def check_committee_consensus(committees: Dict[str, List[Dict]]) -> Dict[str, Any]:
    """Check if all nodes elected the same committee."""
    if not committees:
        return {'consensus': False, 'reason': 'No committee data'}
    
    node_ids = list(committees.keys())
    if len(node_ids) < 2:
        return {'consensus': True, 'reason': 'Only one node'}
    
    first_committee_vks = set(m['vk'] if isinstance(m, dict) else m.get('VK', '') 
                               for m in committees[node_ids[0]])
    
    all_same = True
    for node_id in node_ids[1:]:
        current_vks = set(m['vk'] if isinstance(m, dict) else m.get('VK', '') 
                          for m in committees[node_id])
        if current_vks != first_committee_vks:
            all_same = False
            break
    
    return {
        'consensus': all_same,
        'reason': 'All nodes elected identical committees' if all_same 
                  else 'Committees differ across nodes'
    }

def compute_jaccard_similarity(set1: Set, set2: Set) -> float:
    """Compute Jaccard similarity between two sets."""
    if not set1 and not set2:
        return 1.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0.0

def compute_committee_similarity_matrix(committees: Dict[str, List[Dict]]) -> pd.DataFrame:
    """Compute pairwise Jaccard similarity between all committees."""
    node_ids = list(committees.keys())
    n = len(node_ids)
    similarity_matrix = np.zeros((n, n))
    
    committee_sets = {}
    for node_id in node_ids:
        vks = set(m['vk'] if isinstance(m, dict) else m.get('VK', '') 
                  for m in committees[node_id])
        committee_sets[node_id] = vks
    
    for i, node_i in enumerate(node_ids):
        for j, node_j in enumerate(node_ids):
            similarity_matrix[i, j] = compute_jaccard_similarity(
                committee_sets[node_i], 
                committee_sets[node_j]
            )
    
    return pd.DataFrame(similarity_matrix, index=node_ids, columns=node_ids)

if committees:
    consensus_result = check_committee_consensus(committees)
    
    print("\n=== COMMITTEE CONSENSUS ===")
    print(f"Consensus achieved: {consensus_result['consensus']}")
    print(f"Reason: {consensus_result['reason']}")
else:
    print("\nNo committee data available for consensus analysis.")


## 9. Committee Correlation Analysis

In [ ]:
if committees and len(committees) > 1:
    print("\n=== COMMITTEE CORRELATION ANALYSIS ===")
    print("Computing pairwise committee similarity...")
    
    similarity_matrix = compute_committee_similarity_matrix(committees)
    
    off_diagonal = similarity_matrix.values[np.triu_indices_from(similarity_matrix.values, k=1)]
    
    print(f"\nPairwise Jaccard Similarity Statistics:")
    print(f"Mean: {off_diagonal.mean():.4f}")
    print(f"Std Dev: {off_diagonal.std():.4f}")
    print(f"Min: {off_diagonal.min():.4f}")
    print(f"Max: {off_diagonal.max():.4f}")
    print(f"Median: {np.median(off_diagonal):.4f}")
    
    if len(committees) <= 50:
        plt.figure(figsize=(12, 10))
        sns.heatmap(similarity_matrix, cmap='YlGnBu', vmin=0, vmax=1, 
                    square=True, cbar_kws={'label': 'Jaccard Similarity'})
        plt.title('Committee Similarity Matrix (Jaccard Index)')
        plt.xlabel('Node ID')
        plt.ylabel('Node ID')
        plt.xticks([])
        plt.yticks([])
        plt.tight_layout()
        plt.show()
    else:
        print(f"\nSkipping heatmap visualization ({len(committees)} nodes is too many to display clearly)")
        print("Showing distribution of similarity scores instead...")
        
        plt.figure(figsize=(10, 6))
        plt.hist(off_diagonal, bins=50, edgecolor='black', alpha=0.7)
        plt.xlabel('Jaccard Similarity')
        plt.ylabel('Frequency')
        plt.title('Distribution of Pairwise Committee Similarities')
        plt.axvline(off_diagonal.mean(), color='r', linestyle='--', 
                    label=f'Mean: {off_diagonal.mean():.4f}')
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("\nInsufficient committee data for correlation analysis.")


## 10. Grade Correlation Analysis

In [ ]:
if committees and len(committees) > 1:
    print("\n=== GRADE CORRELATION ANALYSIS ===")
    
    all_vks = set()
    for committee in committees.values():
        for member in committee:
            vk = member['vk'] if isinstance(member, dict) else member.get('VK', '')
            all_vks.add(vk)
    
    grade_matrix = {}
    for node_id, committee in committees.items():
        grades = {}
        for member in committee:
            vk = member['vk'] if isinstance(member, dict) else member.get('VK', '')
            grade = member['grade'] if isinstance(member, dict) else member.get('Grade', 0)
            grades[vk] = grade
        grade_matrix[node_id] = grades
    
    common_vks = set(all_vks)
    for node_id in list(committees.keys())[:10]:
        node_vks = set(grade_matrix[node_id].keys())
        common_vks = common_vks.intersection(node_vks)
    
    if len(common_vks) > 0:
        print(f"Found {len(common_vks)} verification keys present in multiple committees")
        
        node_sample = list(committees.keys())[:min(100, len(committees))]
        correlations = []
        
        for i in range(len(node_sample)):
            for j in range(i + 1, len(node_sample)):
                node_i = node_sample[i]
                node_j = node_sample[j]
                
                shared_vks = set(grade_matrix[node_i].keys()).intersection(
                    set(grade_matrix[node_j].keys())
                )
                
                if len(shared_vks) >= 2:
                    grades_i = [grade_matrix[node_i][vk] for vk in shared_vks]
                    grades_j = [grade_matrix[node_j][vk] for vk in shared_vks]
                    
                    if np.std(grades_i) > 0 and np.std(grades_j) > 0:
                        corr, _ = pearsonr(grades_i, grades_j)
                        correlations.append(corr)
        
        if correlations:
            print(f"\nGrade Correlation Statistics (for shared members):")
            print(f"Mean Pearson correlation: {np.mean(correlations):.4f}")
            print(f"Std Dev: {np.std(correlations):.4f}")
            print(f"Min: {np.min(correlations):.4f}")
            print(f"Max: {np.max(correlations):.4f}")
            
            plt.figure(figsize=(10, 6))
            plt.hist(correlations, bins=30, edgecolor='black', alpha=0.7)
            plt.xlabel('Pearson Correlation Coefficient')
            plt.ylabel('Frequency')
            plt.title('Distribution of Grade Correlations for Shared Committee Members')
            plt.axvline(np.mean(correlations), color='r', linestyle='--', 
                        label=f'Mean: {np.mean(correlations):.4f}')
            plt.legend()
            plt.tight_layout()
            plt.show()
        else:
            print("\nInsufficient shared members to compute grade correlations.")
    else:
        print("\nNo common verification keys found across committees.")
else:
    print("\nInsufficient committee data for grade correlation analysis.")


In [ ]:
print("\n" + "="*60)
print("SIMULATION ANALYSIS SUMMARY")
print("="*60)

print(f"\n📁 Log Directory: {LOG_DIRECTORY}")
print(f"📊 Total Nodes Analyzed: {len(all_logs)}")

print("\n--- GRAPH PROPERTIES ---")
print(f"Nodes: {graph_metrics['num_nodes']}")
print(f"Edges: {graph_metrics['num_edges']}")
print(f"Avg Degree: {graph_metrics['avg_out_degree']:.2f} (out), {graph_metrics['avg_in_degree']:.2f} (in)")
print(f"One-directional edges: {graph_metrics['one_directional_edges']}")
print(f"Strongly connected: {graph_metrics['is_strongly_connected']}")
print(f"Connected components: {graph_metrics['num_strongly_connected_components']}")